# Classical ML + CNN для SER

Признаки: **MFCC** (12 коэф. × 5 статистик), **Pitch/F0** (автокорреляция, 5 статистик),  
**ZCR** (5 статистик), **DWT** (4 уровня × std + skewness + kurtosis).  

Модели: **SVM** (poly kernel), **Random Forest**, **1D CNN**.  
Датасеты: RESD (7 классов) + часть DUSHA (опционально).

## 1. Install

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'librosa', 'pywavelets', 'datasets', 'soundfile',
    'scikit-learn', 'matplotlib', 'seaborn', 'tqdm',
], check=True)
print('Done.')

## 2. Imports

In [ ]:
import os, warnings, pathlib
import numpy as np
import pandas as pd
import librosa
import pywt
import soundfile as sf
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.stats import skew, kurtosis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, classification_report, confusion_matrix,
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3. Load RESD

In [ ]:
from datasets import load_dataset

ds = load_dataset('Aniemore/resd')
print(ds)

RESD_LABEL2ID = {
    'happiness': 0, 'sadness': 1, 'anger': 2,
    'fear': 3, 'disgust': 4, 'enthusiasm': 5, 'neutral': 6,
}
ID2LABEL_RESD = {v: k for k, v in RESD_LABEL2ID.items()}
NUM_CLASSES   = len(RESD_LABEL2ID)

# Собираем (waveform_np, sr, label) из обоих сплитов
SR_TARGET = 16_000
records = []  # list of (waveform np.float32, label_id)

for split in ['train', 'test']:
    for ex in tqdm(ds[split], desc=f'RESD {split}'):
        audio = ex['speech']
        wav   = np.array(audio['array'], dtype=np.float32)
        sr    = audio['sampling_rate']
        if sr != SR_TARGET:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR_TARGET)
        label = RESD_LABEL2ID[ex['emotion']]
        records.append({'wav': wav, 'label': label, 'source': split})

print(f'\nTotal records: {len(records)}')
from collections import Counter
cnt = Counter(r['label'] for r in records)
for lid, n in sorted(cnt.items()):
    print(f'  {ID2LABEL_RESD[lid]:12s}  {n}')

## 4. (Опционально) Добавить часть DUSHA

Маппинг DUSHA → RESD: `neutral→neutral`, `angry→anger`, `positive→happiness`, `sad→sadness`.  
Класс `other` пропускается.

In [ ]:
USE_DUSHA = False  # ← поставь True на Kaggle

DUSHA_TSV       = '/kaggle/input/datasets/aleksandribryanov/agg-dusha/aggregated_majority.tsv'
DUSHA_AUDIO_DIR = '/kaggle/input/datasets/sigireddybalasai/dusha-datasetcrowd/crowd_train'
DUSHA_FRACTION  = 0.1   # доля датасета (10%)

DUSHA2RESD = {
    'neutral':  RESD_LABEL2ID['neutral'],
    'angry':    RESD_LABEL2ID['anger'],
    'positive': RESD_LABEL2ID['happiness'],
    'sad':      RESD_LABEL2ID['sadness'],
}

if USE_DUSHA and pathlib.Path(DUSHA_TSV).exists():
    df = pd.read_csv(DUSHA_TSV, sep='\t')
    df = df[df['aggregated_emo'].isin(DUSHA2RESD)]
    df = df.sample(frac=DUSHA_FRACTION, random_state=SEED)
    added = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc='DUSHA'):
        path = pathlib.Path(DUSHA_AUDIO_DIR) / row['audio_path']
        if not path.exists():
            continue
        wav, sr = sf.read(str(path), dtype='float32')
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        if sr != SR_TARGET:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR_TARGET)
        records.append({'wav': wav, 'label': DUSHA2RESD[row['aggregated_emo']], 'source': 'dusha'})
        added += 1
    print(f'Added {added} DUSHA samples')
else:
    print('DUSHA disabled — using RESD only')

## 5. Feature Extraction

In [ ]:
N_MFCC      = 12
DWT_WAVELET = 'db4'
DWT_LEVELS  = 4
HOP_LENGTH  = 512
N_FFT       = 2048

def _stats(arr):
    """[min, max, mean, median, std]"""
    return [float(np.min(arr)), float(np.max(arr)),
            float(np.mean(arr)), float(np.median(arr)), float(np.std(arr))]


def extract_mfcc(wav, sr=SR_TARGET):
    """12 MFCC coefficients × 5 stats = 60 features."""
    mfcc = librosa.feature.mfcc(y=wav, sr=sr, n_mfcc=N_MFCC,
                                 n_fft=N_FFT, hop_length=HOP_LENGTH)
    feats = []
    for coef in mfcc:
        feats.extend(_stats(coef))
    return feats  # 60


def extract_pitch(wav, sr=SR_TARGET, frame_length=2048, hop=512):
    """
    F0 через автокорреляцию.
    Для каждого окна: F0 = sr / lag_max_autocorr  (диапазон 50–500 Hz).
    Возвращает 5 агрегированных статистик.
    """
    min_lag = int(sr / 500)   # 500 Hz верхняя граница
    max_lag = int(sr / 50)    # 50 Hz нижняя граница
    f0s = []
    for start in range(0, max(1, len(wav) - frame_length), hop):
        frame = wav[start:start + frame_length]
        if len(frame) < frame_length:
            break
        # автокорреляция
        corr = np.correlate(frame, frame, mode='full')
        corr = corr[len(corr) // 2:]          # берём правую половину
        lag_range = corr[min_lag:min(max_lag, len(corr))]
        if len(lag_range) == 0:
            continue
        peak_lag = np.argmax(lag_range) + min_lag
        f0s.append(sr / peak_lag if peak_lag > 0 else 0.0)
    if not f0s:
        f0s = [0.0]
    return _stats(np.array(f0s))  # 5


def extract_zcr(wav, sr=SR_TARGET):
    """Zero Crossing Rate — 5 stats."""
    zcr = librosa.feature.zero_crossing_rate(wav, hop_length=HOP_LENGTH)[0]
    return _stats(zcr)  # 5


def extract_dwt(wav):
    """
    DWT (db4, 4 уровня).
    На каждом уровне (включая аппроксимацию): std, skewness, kurtosis.
    Итого (4+1) × 3 = 15 признаков.
    """
    coeffs = pywt.wavedec(wav, DWT_WAVELET, level=DWT_LEVELS)
    feats = []
    for c in coeffs:
        feats.extend([
            float(np.std(c)),
            float(skew(c)),
            float(kurtosis(c)),
        ])
    return feats  # 15


def extract_features(wav, sr=SR_TARGET):
    """Полный вектор признаков: 60+5+5+15 = 85 features."""
    return (
        extract_mfcc(wav, sr)
        + extract_pitch(wav, sr)
        + extract_zcr(wav, sr)
        + extract_dwt(wav)
    )


# имена признаков для RF importance
feature_names = []
for i in range(N_MFCC):
    for s in ['min', 'max', 'mean', 'median', 'std']:
        feature_names.append(f'mfcc_{i:02d}_{s}')
for s in ['min', 'max', 'mean', 'median', 'std']:
    feature_names.append(f'pitch_{s}')
for s in ['min', 'max', 'mean', 'median', 'std']:
    feature_names.append(f'zcr_{s}')
for lvl in range(DWT_LEVELS + 1):
    for s in ['std', 'skew', 'kurt']:
        feature_names.append(f'dwt_L{lvl}_{s}')

N_FEATURES = len(feature_names)
print(f'Feature vector size: {N_FEATURES}')  # 85

In [ ]:
print('Extracting features...')
X_list, y_list = [], []

for r in tqdm(records):
    try:
        feats = extract_features(r['wav'])
        X_list.append(feats)
        y_list.append(r['label'])
    except Exception as e:
        pass  # skip broken audio

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int64)
print(f'X shape: {X.shape}  y shape: {y.shape}')

In [ ]:
# train/test split стратифицированный
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

LABEL_NAMES = [ID2LABEL_RESD[i] for i in range(NUM_CLASSES)]
print(f'Train: {len(X_train)}  Test: {len(X_test)}')

## 6. SVM (polynomial kernel)

In [ ]:
from sklearn.model_selection import GridSearchCV

svm = SVC(kernel='poly', degree=3, C=1.0, coef0=1.0,
          class_weight='balanced', random_state=SEED)
svm.fit(X_train_s, y_train)

svm_pred = svm.predict(X_test_s)
print('=== SVM (poly, d=3) ===')
print(f'Accuracy          : {accuracy_score(y_test, svm_pred):.4f}')
print(f'Weighted Accuracy  : {balanced_accuracy_score(y_test, svm_pred):.4f}')
print(f'F1 Macro          : {f1_score(y_test, svm_pred, average="macro", zero_division=0):.4f}')
print(f'F1 Weighted       : {f1_score(y_test, svm_pred, average="weighted", zero_division=0):.4f}')
print()
print(classification_report(y_test, svm_pred, target_names=LABEL_NAMES, zero_division=0))

## 7. Random Forest + Feature Importance

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=None,
    class_weight='balanced', random_state=SEED, n_jobs=-1,
)
rf.fit(X_train_s, y_train)

rf_pred = rf.predict(X_test_s)
print('=== Random Forest ===')
print(f'Accuracy          : {accuracy_score(y_test, rf_pred):.4f}')
print(f'Weighted Accuracy  : {balanced_accuracy_score(y_test, rf_pred):.4f}')
print(f'F1 Macro          : {f1_score(y_test, rf_pred, average="macro", zero_division=0):.4f}')
print(f'F1 Weighted       : {f1_score(y_test, rf_pred, average="weighted", zero_division=0):.4f}')
print()
print(classification_report(y_test, rf_pred, target_names=LABEL_NAMES, zero_division=0))

In [ ]:
# Feature importance — топ-30
importances = rf.feature_importances_
idx = np.argsort(importances)[::-1][:30]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh([feature_names[i] for i in idx[::-1]], importances[idx[::-1]], color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('Random Forest — Top 30 Feature Importances')
plt.tight_layout()
plt.savefig('/kaggle/working/rf_feature_importance.png', dpi=150)
plt.show()

## 8. CNN

Входные данные: MFCC последовательность **(T=216 фреймов × 12 коэф.)**, паддинг/обрезка до фиксированной длины.  
Архитектура: 6 Conv1D блоков → MaxPool → Flatten → Dense.

In [ ]:
MAX_FRAMES = 216   # фреймов MFCC (≈ 6.9 с при hop=512, sr=16000)

def extract_mfcc_sequence(wav, sr=SR_TARGET, max_frames=MAX_FRAMES):
    """Возвращает np.array shape (max_frames, N_MFCC), с паддингом/обрезкой."""
    mfcc = librosa.feature.mfcc(y=wav, sr=sr, n_mfcc=N_MFCC,
                                 n_fft=N_FFT, hop_length=HOP_LENGTH)  # (12, T)
    mfcc = mfcc.T  # (T, 12)
    T = mfcc.shape[0]
    if T >= max_frames:
        return mfcc[:max_frames]
    pad = np.zeros((max_frames - T, N_MFCC), dtype=np.float32)
    return np.vstack([mfcc, pad])


class MFCCDataset(Dataset):
    def __init__(self, records, max_frames=MAX_FRAMES):
        self.data = []
        for r in tqdm(records, desc='MFCC sequences'):
            try:
                seq = extract_mfcc_sequence(r['wav'], max_frames=max_frames)
                self.data.append((seq.astype(np.float32), r['label']))
            except:
                pass

    def __len__(self):  return len(self.data)
    def __getitem__(self, i):
        x, y = self.data[i]
        return torch.tensor(x), torch.tensor(y, dtype=torch.long)


# сплит как у классических моделей (воспроизводим тот же random state)
train_recs, test_recs = train_test_split(
    records, test_size=0.2, random_state=SEED,
    stratify=[r['label'] for r in records])

train_ds = MFCCDataset(train_recs)
test_ds  = MFCCDataset(test_recs)
print(f'CNN train: {len(train_ds)}  test: {len(test_ds)}')

In [ ]:
class AudioCNN(nn.Module):
    """
    Архитектура из скриншота, адаптирована для (MAX_FRAMES, N_MFCC) входа.
    В PyTorch Conv1d работает на (batch, channels, length),
    поэтому делаем permute (B, T, C) → (B, C, T).
    """
    def __init__(self, n_mfcc=N_MFCC, max_frames=MAX_FRAMES,
                 num_classes=NUM_CLASSES, dropout=0.25):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(n_mfcc, 128, kernel_size=6, padding='same'),
            nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=5, padding='same'),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.MaxPool1d(8),          # MAX_FRAMES → MAX_FRAMES//8
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(128, 128, kernel_size=5, padding='same'),
            nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=5, padding='same'),
            nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=5, padding='same'),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(128, 128, kernel_size=5, padding='same'),
            nn.ReLU(),
        )
        out_len = max_frames // 8      # 27 при MAX_FRAMES=216
        self.classifier = nn.Linear(out_len * 128, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)        # (B, T, C) → (B, C, T)
        x = self.block1(x)
        x = self.block2(x)
        x = x.flatten(1)
        return self.classifier(x)


model = AudioCNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')

In [ ]:
BATCH_SIZE = 32
EPOCHS     = 60
LR         = 1e-3
ES_PATIENCE = 10   # early stopping patience

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, verbose=True)

# --- Early Stopping ---
class EarlyStopping:
    def __init__(self, patience=10, path='best_cnn.pt'):
        self.patience = patience
        self.path     = path
        self.best     = -1.0
        self.counter  = 0
        self.stop     = False

    def step(self, metric, model):
        if metric > self.best:
            self.best    = metric
            self.counter = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

es = EarlyStopping(patience=ES_PATIENCE, path='/kaggle/working/best_cnn.pt')

history = {'train_loss': [], 'val_wacc': [], 'lr': []}

for epoch in range(1, EPOCHS + 1):
    # --- train ---
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)
    avg_loss = total_loss / len(train_ds)

    # --- val ---
    model.eval()
    preds_list, labels_list = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            preds_list.append(model(xb.to(device)).argmax(1).cpu().numpy())
            labels_list.append(yb.numpy())
    val_preds  = np.concatenate(preds_list)
    val_labels = np.concatenate(labels_list)
    wacc = balanced_accuracy_score(val_labels, val_preds)

    current_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(avg_loss)
    history['val_wacc'].append(wacc)
    history['lr'].append(current_lr)

    scheduler.step(wacc)
    es.step(wacc, model)

    print(f'Epoch {epoch:3d}/{EPOCHS}  loss={avg_loss:.4f}  '
          f'val_wacc={wacc:.4f}  lr={current_lr:.2e}'
          + ('  *' if es.counter == 0 else ''), flush=True)

    if es.stop:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nBest val_wacc: {es.best:.4f}')

In [ ]:
# Кривые обучения
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'])
ax1.set_title('Train Loss')
ax1.set_xlabel('Epoch')

ax2.plot(history['val_wacc'], color='darkorange')
ax2.axhline(es.best, color='red', linestyle='--', label=f'best={es.best:.4f}')
ax2.set_title('Val Weighted Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.savefig('/kaggle/working/cnn_training_curves.png', dpi=150)
plt.show()

In [ ]:
# Финальная оценка CNN (best checkpoint)
model.load_state_dict(torch.load('/kaggle/working/best_cnn.pt', map_location=device))
model.eval()

preds_list, labels_list = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds_list.append(model(xb.to(device)).argmax(1).cpu().numpy())
        labels_list.append(yb.numpy())

cnn_pred   = np.concatenate(preds_list)
cnn_labels = np.concatenate(labels_list)

print('=== CNN ===')
print(f'Accuracy          : {accuracy_score(cnn_labels, cnn_pred):.4f}')
print(f'Weighted Accuracy  : {balanced_accuracy_score(cnn_labels, cnn_pred):.4f}')
print(f'F1 Macro          : {f1_score(cnn_labels, cnn_pred, average="macro", zero_division=0):.4f}')
print(f'F1 Weighted       : {f1_score(cnn_labels, cnn_pred, average="weighted", zero_division=0):.4f}')
print()
print(classification_report(cnn_labels, cnn_pred, target_names=LABEL_NAMES, zero_division=0))

## 9. Confusion Matrix — CNN

In [ ]:
cm      = confusion_matrix(cnn_labels, cnn_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[0])
axes[0].set_title('CNN Confusion Matrix (counts)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[1])
axes[1].set_title('CNN Confusion Matrix (row-normalized)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig('/kaggle/working/cnn_confusion_matrix.png', dpi=150)
plt.show()

## 10. Сравнение моделей

In [ ]:
results = {
    'SVM (poly)': svm_pred,
    'Random Forest': rf_pred,
    'CNN': cnn_pred,
}
ref_labels = {'SVM (poly)': y_test, 'Random Forest': y_test, 'CNN': cnn_labels}

rows = []
for name, pred in results.items():
    labels_ref = ref_labels[name]
    rows.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(labels_ref, pred), 4),
        'WAcc':      round(balanced_accuracy_score(labels_ref, pred), 4),
        'F1 Macro':  round(f1_score(labels_ref, pred, average='macro',    zero_division=0), 4),
        'F1 Weighted': round(f1_score(labels_ref, pred, average='weighted', zero_division=0), 4),
    })

df_res = pd.DataFrame(rows).set_index('Model')
print(df_res.to_string())
df_res.to_csv('/kaggle/working/results_comparison.csv')